# Fine-tuning Médical — TechCorp Hackathon
**Modèle :** microsoft/Phi-3.5-mini-instruct  
**Dataset :** `medical_dataset_final.json` (4 000 exemples nettoyés par l'équipe DATA)  
**Technique :** QLoRA (4-bit + LoRA)  
**Équipe :** Inacio RODRIGUES, Théo SALY, Olivier DICK, Hayat MEGHLAT

> ⚠️ Ce modèle est **expérimental** — pas destiné à la production médicale.

**Activer le GPU :** Runtime → Changer le type d'exécution → GPU T4

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes datasets trl

In [ ]:
# Upload du dataset médical nettoyé depuis votre machine
from google.colab import files
print('Uploadez le fichier datasets/medical_dataset_final.json')
uploaded = files.upload()
DATASET_PATH = 'medical_dataset_final.json'

In [ ]:
import torch, json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
from trl import SFTTrainer

print(f'GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NON DISPONIBLE"}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

In [ ]:
MODEL_NAME  = 'microsoft/Phi-3.5-mini-instruct'
OUTPUT_DIR  = './phi35-medical-lora'
MAX_SEQ_LEN = 512
EPOCHS      = 2
BATCH_SIZE  = 2
GRAD_ACCUM  = 4
LR          = 2e-4

In [ ]:
# Chargement du dataset médical nettoyé (format instruction/input/output)
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)
print(f'Exemples chargés : {len(raw)}')
print('Exemple :', raw[0])

In [ ]:
# Formatage au format Phi-3.5 chat
def format_sample(item):
    question = item.get('input', item.get('Patient', item.get('question', '')))
    answer   = item.get('output', item.get('Doctor',  item.get('answer', '')))
    return {
        'text': (
            '<|system|>\nYou are a helpful medical assistant. '
            'Always remind the user to consult a qualified doctor for diagnosis and treatment.<|end|>\n'
            f'<|user|>\n{question}<|end|>\n'
            f'<|assistant|>\n{answer}<|end|>'
        )
    }

dataset = Dataset.from_list([format_sample(i) for i in raw])
print(f'Dataset formaté : {len(dataset)} exemples')
print(dataset[0]['text'][:400])

In [ ]:
# Chargement du modèle en 4-bit (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print('Modèle chargé.')

In [ ]:
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=['qkv_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
trainable, total = model.get_nb_trainable_parameters()
print(f'Paramètres entraînables : {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=50,
    logging_steps=25,
    save_steps=200,
    save_total_limit=2,
    fp16=True,
    report_to='none',
)

trainer = SFTTrainer(
    model=model, train_dataset=dataset,
    args=training_args, tokenizer=tokenizer,
    dataset_text_field='text', max_seq_length=MAX_SEQ_LEN,
)

print('Démarrage de l\'entraînement...')
trainer.train()
print('Entraînement terminé !')

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Modèle sauvegardé dans {OUTPUT_DIR}')

In [ ]:
model.eval()
TEST_QUESTIONS = [
    'I have a headache and fever since 2 days. What should I do?',
    'What are the symptoms of diabetes?',
    'How can I lower my blood pressure naturally?',
]
for question in TEST_QUESTIONS:
    prompt = f'<|system|>\nYou are a helpful medical assistant.<|end|>\n<|user|>\n{question}<|end|>\n<|assistant|>\n'
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f'\n❓ {question}')
    print(f'💬 {response.strip()}')

## Métriques d'entraînement

| Métrique | Valeur |
|---|---|
| Modèle de base | Phi-3.5-mini-instruct |
| Dataset | medical_dataset_final.json |
| Samples | 4 000 |
| Epochs | 2 |
| Loss initiale | _à compléter_ |
| Loss finale | _à compléter_ |
| Durée | _à compléter_ |
| GPU | T4 |

⚠️ Ce modèle est expérimental et ne remplace pas l'avis d'un médecin.